 # MetaMARL visualization module tutorial



 This file is a **VS Code / Jupyter percent-format notebook**.



 Save it as:



 `visualization_module_tutorial.py`



 VS Code recognizes every `# %%` marker as a notebook cell. You can run cells

 directly, or use the Command Palette:



 **Jupyter: Export Current Python File as Jupyter Notebook**



 Do **not** rename a `.txt` file to `.ipynb`: an `.ipynb` file is JSON, not

 plain Python. The percent-format `.py` file is the easiest copy-paste-ready

 source format for this tutorial.



 ---



 ## Goal



 The visualization stack separates four concerns:



 ```

 WHAT DATA EXISTS        -> MetricSchema

 HOW DATA ACCUMULATES    -> Metric / MetricLogger

 WHAT DATA TO SELECT     -> Query

 HOW IT IS DISPLAYED     -> Reporter

 ```



 The important consequence is that the environment, inner optimizer, and

 outer optimizer do not need W&B-specific plotting logic. They expose typed

 metrics. Queries select paths from those metrics. A reporter renders the

 selected data to W&B, CSV, TensorBoard, or another backend.



 This tutorial documents:



 1. how schemas, logger, queries, and reporters fit together;

 2. where environment, Ray/RLlib, and ES queries are configured;

 3. how to add new metrics by subclassing schemas at the correct level;

 4. how to use `peek()` for accumulated series without destroying history;

 5. the exact fishery query sets that should be used for visualization

    regression testing;

 6. the target wildcard query syntax required for runtime mechanism, seed,

    episode, policy, agent, and parameter IDs;

 7. the current limitations that still need implementation.

In [ ]:
from __future__ import annotations

# These imports are the current branch locations used by the tutorial.
# Adjust only if the package layout changes.
from core.optimizers.es.schema import ESSchema
from core.reporting.query import Query


 ## 1. Query paths are tuples



 A query points to a metric leaf through a schema tree.



 **Important Python syntax:**



 ```python

 ("generation")   # string -- WRONG

 ("generation",)  # one-element tuple -- CORRECT

 ```



 The current `Query` API has the conceptual shape:



 ```python

 Query(

     title: str,

     x: tuple[str, ...],

     y: tuple[str, ...] | tuple[tuple[str, ...], ...],

     reduce: Literal["none", "mean"] = "none",

     error: Literal["none", "std"] = "none",

 )

 ```



 Semantics:



 - one `y` path -> one series;

 - multiple `y` paths + `reduce="none"` -> multiple raw series;

 - multiple `y` paths + `reduce="mean"` -> pointwise mean;

 - `reduce="mean", error="std"` -> pointwise mean with ±1 standard deviation;

 - `error="std"` is only meaningful when data is reduced across multiple

   matched series.



 At the time of this tutorial, paths with runtime dictionary keys still

 require concrete IDs. Wildcard support such as `"*"` is a TODO documented

 later.

In [ ]:
simple_query = Query(
    title="Fish biomass",
    x=("iter",),
    y=("fish_norm",),
)


 ## 2. Metric accumulation: `peek()` versus `reduce()`



 `MetricLogger` owns the accumulated metric state.



 Use:



 ```python

 logger.push_data(payload)

 current = logger.peek()

 reporter.report(current)

 ```



 when you want to render the complete accumulated history without destroying

 it.



 Use:



 ```python

 reduced = logger.reduce()

 ```



 when you want the configured reducers to compile the accumulated data and

 clear the underlying metric state.



 This distinction matters especially for ES. ES plots are cumulative over

 outer generations. Therefore ES reporting should use `peek()`, not

 `reduce()`, between generations.

 ### ES reporting pattern



 The intended outer-loop pattern is:



 ```python

 metrics = self._to_logger_payload(

     inner=info["metrics"],

     population=population,

     fitness=fitness,

     mean=pre_update_mean,

     sigma=pre_update_sigma,

 )



 self.logger.push_data(metrics)



 accumulated_metrics = self.logger.peek()

 self.reporting.report(accumulated_metrics)

 ```



 Generation 1:



 ```

 generation    -> [1]

 fitness_mean  -> [f1]

 ```



 Generation 2:



 ```

 generation    -> [1, 2]

 fitness_mean  -> [f1, f2]

 ```



 Generation 3:



 ```

 generation    -> [1, 2, 3]

 fitness_mean  -> [f1, f2, f3]

 ```



 The reporter receives the complete trajectory each time. It does not need

 to own a second history cache.

 ## 3. Where reporting is configured



 There are three distinct reporting levels in a bilevel run.



 ### A. Environment-level reporting



 Configure the environment schema and its horizon queries in `.environment`:



 ```python

 .environment(

     env=FisheryRegulatedEnv,

     ...,

     schema=FisheryMetricSchema,

     queries=FISHERY_ENV_QUERIES,

 )

 ```



 These queries operate directly on one `FisheryMetricSchema` and are ideal for

 environment-horizon plots.



 ### B. Inner optimizer reporting



 Configure the inner optimizer result schema in `.reporting`:



 ```python

 .reporting(

     schema=RaySchema,

     queries=RAY_QUERIES,

 )

 ```



 These queries operate on RLlib training/evaluation results: rollout

 aggregates, by-mechanism/by-seed/by-episode data, learner metrics, and

 performance metrics.



 ### C. ES / outer optimizer reporting



 Configure ES in the outer optimizer:



 ```python

 .reporting(

     schema=ESSchema,

     queries=ES_QUERIES,

 )

 ```



 ES stores cumulative SERIES values across outer generations.



 A typical bilevel configuration therefore has a reporter backend at the

 bilevel level and separate query sets attached to environment, inner

 optimizer, and outer optimizer schemas.

 ## 4. Schema hierarchy and where to add new metrics



 The schema level determines the meaning and path of a metric.



 ```

 MetricSchema

 │

 ├── Environment

 │   ├── EpisodeRolloutSchema

 │   │   └── by_agent

 │   │       └── AgentEnvStepSchema

 │   │

 │   └── FisheryMetricSchema(EpisodeRolloutSchema)

 │       └── by_agent

 │           └── FisheryAgentMetricSchema(AgentEnvStepSchema)

 │

 ├── RaySchema

 │   ├── train

 │   │   ├── rollout

 │   │   │   ├── aggregate

 │   │   │   └── by_mechanism

 │   │   │       └── by_seed

 │   │   │           └── by_episode

 │   │   ├── learner

 │   │   │   └── by_policy

 │   │   └── performance

 │   └── eval

 │       ├── rollout

 │       └── performance

 │

 └── ESSchema

     ├── ES-level SERIES

     ├── by_mechanism

     │   └── by_parameter

     ├── search_mean

     ├── global_best

     └── inner: MetricSchema

         └── runtime subtype, e.g. RaySchema

 ```



 The logger supports runtime schema specialization. For example:



 ```

 EpisodeRolloutSchema -> FisheryMetricSchema

 AgentEnvStepSchema    -> FisheryAgentMetricSchema

 MetricSchema          -> RaySchema   (ESSchema.inner)

 ```



 The runtime subtype must remain compatible with the declared base schema.

 ### 4.1 Add a new environment-level metric



 If the metric describes the shared environment/episode, subclass

 `EpisodeRolloutSchema`.



 Example:



 ```python

 class FisheryMetricSchema(EpisodeRolloutSchema):

     fish_norm: Optional[float] = Field(

         default=None,

         json_schema_extra={"reduce": ReduceProtocol.MEAN},

     )

 ```



 The query path is then:



 ```python

 ("fish_norm",)

 ```



 or, once nested inside Ray:



 ```python

 (

     "train",

     "rollout",

     "by_mechanism",

     mechanism_id,

     "by_seed",

     seed_id,

     "by_episode",

     episode_id,

     "fish_norm",

 )

 ```

 ### 4.2 Add a new per-agent environment metric



 If the value differs by agent, subclass `AgentEnvStepSchema`, then override

 `by_agent` in the environment-specific episode schema.



 ```python

 class FisheryAgentMetricSchema(AgentEnvStepSchema):

     requested_harvest: Optional[float] = Field(

         default=None,

         json_schema_extra={"reduce": ReduceProtocol.MEAN},

     )





 class FisheryMetricSchema(EpisodeRolloutSchema):

     by_agent: dict[str, FisheryAgentMetricSchema] = Field(

         default_factory=dict,

     )

 ```



 Query path:



 ```python

 ("by_agent", "utilizer:0", "requested_harvest")

 ```



 Target wildcard path after dynamic Query support:



 ```python

 ("by_agent", "*", "requested_harvest")

 ```

 ### 4.3 Add a new learner metric



 Add it to `PolicyLearnerSchema`.



 Example:



 ```python

 class PolicyLearnerSchema(MetricSchema):

     gradient_norm: Optional[float] = Field(

         default=None,

         json_schema_extra={"reduce": ReduceProtocol.MEAN},

     )

 ```



 Concrete query today:



 ```python

 (

     "train",

     "learner",

     "by_policy",

     policy_id,

     "gradient_norm",

 )

 ```



 Target wildcard:



 ```python

 ("train", "learner", "by_policy", "*", "gradient_norm")

 ```

 ### 4.4 Add a new performance metric



 Add it to `PerformanceSchema`.



 Example path:



 ```python

 ("train", "performance", "env_steps_throughput")

 ```

 ### 4.5 Add a new ES metric



 ES values that must be plotted across outer generations should use

 `ReduceProtocol.SERIES`.



 Example:



 ```python

 class ESSchema(MetricSchema):

     sigma: Optional[float] = Field(

         default=None,

         json_schema_extra={"reduce": ReduceProtocol.SERIES},

     )

 ```



 The ES logger stays alive across outer iterations, so the series grows with

 each call to `push_data()`.

 ### 4.6 IMPORTANT: verify `generation` exists in `ESSchema`



 The current ES optimizer creates:



 ```python

 ESSchema(generation=self.generation, ...)

 ```



 but the schema pasted with this feature work did not show an explicit

 `generation` field.



 If `ESSchema` does not already declare it, add:



 ```python

 generation: Optional[int] = Field(

     default=None,

     json_schema_extra={"reduce": ReduceProtocol.SERIES},

 )

 ```



 Otherwise use the inherited `iter` field consistently instead. Do not have

 the optimizer write `generation` while queries read `iter`, or vice versa.

In [ ]:
if "generation" not in ESSchema.model_fields:
    print(
        "NOTE: ESSchema currently has no explicit 'generation' field. "
        "Add generation as SERIES or use 'iter' consistently."
    )


 ### 4.7 `ESSchema.inner` is intentionally generic



 Keep:



 ```python

 inner: Optional[MetricSchema] = None

 ```



 rather than hard-coding `RaySchema`.



 The logger late-binds the runtime subtype:



 ```

 MetricSchema -> RaySchema

 ```



 This allows ES to wrap another optimizer implementation without changing the

 ES schema.

 ## 5. Environment-horizon queries: fishery



 These queries operate directly on `FisheryMetricSchema`.



 They are the first set that should be regression-tested against the dev

 environment horizon plots.

In [ ]:
FISHERY_ENV_QUERIES = [
    Query(
        title="Fish biomass",
        x=("iter",),
        y=("fish_norm",),
    ),
    Query(
        title="Next fish biomass",
        x=("iter",),
        y=("fish_norm_next",),
    ),
    Query(
        title="Fish stock",
        x=("iter",),
        y=("fish_stock",),
    ),
    Query(
        title="Next fish stock",
        x=("iter",),
        y=("fish_stock_next",),
    ),
    Query(
        title="Biological growth",
        x=("iter",),
        y=("growth",),
    ),
    Query(
        title="Growth noise",
        x=("iter",),
        y=("growth_noise",),
    ),
    Query(
        title="Attempted harvest",
        x=("iter",),
        y=("H_attempted",),
    ),
    Query(
        title="Realized harvest",
        x=("iter",),
        y=("H_realized",),
    ),
    Query(
        title="Allowed harvest",
        x=("iter",),
        y=("allowed_harvest",),
    ),
    Query(
        title="Total usage normalized",
        x=("iter",),
        y=("total_usage_norm",),
    ),
    Query(
        title="Quota stress",
        x=("iter",),
        y=("quota_stress",),
    ),
    Query(
        title="Biomass at MSY",
        x=("iter",),
        y=("B_msy",),
    ),
    Query(
        title="Maximum sustainable yield",
        x=("iter",),
        y=("MSY",),
    ),
    Query(
        title="Fishing mortality at MSY",
        x=("iter",),
        y=("F_msy",),
    ),
]


 ### Environment metrics that existed in dev plotting but are not shown in

 ### the provided `FisheryMetricSchema`



 The old dev environment plotting code also referenced fields such as:



 - `full_required_harvest`

 - `min_demand_frac`

 - `max_demand_frac`

 - environment-level `intrinsic_utility`



 If exact plot parity requires those quantities, add them to

 `FisheryMetricSchema` (if shared) or `FisheryAgentMetricSchema` (if

 agent-specific) before adding queries. Do not create a query for a field that

 is not represented in the schema.

 ## 6. Per-agent environment queries



 The current Query API requires the concrete runtime agent ID.



 Use a small helper today:

In [ ]:
def fishery_agent_queries(agent_id: str) -> list[Query]:
    base = ("by_agent", agent_id)
    return [
        Query(
            title=f"Reward — {agent_id}",
            x=("iter",),
            y=base + ("reward",),
        ),
        Query(
            title=f"Action — {agent_id}",
            x=("iter",),
            y=base + ("action",),
        ),
        Query(
            title=f"Observation — {agent_id}",
            x=("iter",),
            y=base + ("observation",),
        ),
        Query(
            title=f"Intrinsic utility — {agent_id}",
            x=("iter",),
            y=base + ("intrinsic_utility",),
        ),
        Query(
            title=f"Violation signal — {agent_id}",
            x=("iter",),
            y=base + ("violation_signal",),
        ),
        Query(
            title=f"Requested harvest — {agent_id}",
            x=("iter",),
            y=base + ("requested_harvest",),
        ),
        Query(
            title=f"Delivered harvest — {agent_id}",
            x=("iter",),
            y=base + ("delivered_harvest",),
        ),
        Query(
            title=f"Requested harvest fraction — {agent_id}",
            x=("iter",),
            y=base + ("requested_frac",),
        ),
        Query(
            title=f"Quota violation — {agent_id}",
            x=("iter",),
            y=base + ("quota_violation",),
        ),
        Query(
            title=f"Quota penalty — {agent_id}",
            x=("iter",),
            y=base + ("quota_penalty",),
        ),
        Query(
            title=f"Risk penalty — {agent_id}",
            x=("iter",),
            y=base + ("risk_penalty",),
        ),
    ]


EXAMPLE_AGENT_QUERIES = fishery_agent_queries("utilizer:0")


 Target API after wildcard support:



 ```python

 Query(

     title="Reward by agent",

     x=("iter",),

     y=("by_agent", "*", "reward"),

 )

 ```



 Expected result: one trace for every runtime agent key, with deterministic

 labels derived from the matched IDs.

 ## 7. Inner optimizer: Ray/RLlib aggregate queries



 These queries operate on `RaySchema`.



 The inherited root `iter` is the RLlib training iteration and is the natural

 x-axis for raw RLlib rollout, learner, and performance metrics.

In [ ]:
RAY_ROLLOUT_QUERIES = [
    Query(
        title="Train reward",
        x=("iter",),
        y=(
            ("train", "rollout", "aggregate", "reward_mean"),
            ("train", "rollout", "aggregate", "reward_min"),
            ("train", "rollout", "aggregate", "reward_max"),
        ),
    ),
    Query(
        title="Train episode length",
        x=("iter",),
        y=(
            ("train", "rollout", "aggregate", "episode_len_mean"),
            ("train", "rollout", "aggregate", "episode_len_min"),
            ("train", "rollout", "aggregate", "episode_len_max"),
        ),
    ),
    Query(
        title="Train episodes",
        x=("iter",),
        y=("train", "rollout", "aggregate", "num_episodes"),
    ),
    Query(
        title="Train episodes lifetime",
        x=("iter",),
        y=("train", "rollout", "aggregate", "num_episodes_lifetime"),
    ),
    Query(
        title="Eval reward",
        x=("iter",),
        y=(
            ("eval", "rollout", "aggregate", "reward_mean"),
            ("eval", "rollout", "aggregate", "reward_min"),
            ("eval", "rollout", "aggregate", "reward_max"),
        ),
    ),
    Query(
        title="Eval episode length",
        x=("iter",),
        y=(
            ("eval", "rollout", "aggregate", "episode_len_mean"),
            ("eval", "rollout", "aggregate", "episode_len_min"),
            ("eval", "rollout", "aggregate", "episode_len_max"),
        ),
    ),
    Query(
        title="Eval episodes",
        x=("iter",),
        y=("eval", "rollout", "aggregate", "num_episodes"),
    ),
]


 ## 8. Inner optimizer: performance queries

In [ ]:
RAY_PERFORMANCE_QUERIES = [
    Query(
        title="Train environment steps",
        x=("iter",),
        y=(
            ("train", "performance", "env_steps_this_iter"),
            ("train", "performance", "env_steps_lifetime"),
        ),
    ),
    Query(
        title="Train agent steps",
        x=("iter",),
        y=(
            ("train", "performance", "agent_steps_this_iter_sum"),
            ("train", "performance", "agent_steps_lifetime_sum"),
        ),
    ),
    Query(
        title="Environment throughput",
        x=("iter",),
        y=("train", "performance", "env_steps_throughput"),
    ),
    Query(
        title="Training timing",
        x=("iter",),
        y=(
            ("train", "performance", "training_iteration_s"),
            ("train", "performance", "training_step_s"),
            ("train", "performance", "sample_s"),
            ("train", "performance", "learner_update_s"),
        ),
    ),
    Query(
        title="Weights sequence number",
        x=("iter",),
        y=("train", "performance", "weights_seq_no"),
    ),
    Query(
        title="Eval environment steps",
        x=("iter",),
        y=(
            ("eval", "performance", "env_steps_this_iter"),
            ("eval", "performance", "env_steps_lifetime"),
        ),
    ),
    Query(
        title="Eval agent steps",
        x=("iter",),
        y=(
            ("eval", "performance", "agent_steps_this_iter_sum"),
            ("eval", "performance", "agent_steps_lifetime_sum"),
        ),
    ),
    Query(
        title="Eval weights sequence number",
        x=("iter",),
        y=("eval", "performance", "weights_seq_no"),
    ),
]


 ## 9. Inner optimizer: per-policy learner queries



 Policy IDs are runtime-defined. The current API therefore needs a concrete

 policy ID.

In [ ]:
def ray_policy_queries(policy_id: str) -> list[Query]:
    base = ("train", "learner", "by_policy", policy_id)

    return [
        Query(
            title=f"Batch size — {policy_id}",
            x=("iter",),
            y=base + ("batch_size",),
        ),
        Query(
            title=f"Total loss — {policy_id}",
            x=("iter",),
            y=base + ("total_loss",),
        ),
        Query(
            title=f"Residual variance — {policy_id}",
            x=("iter",),
            y=base + ("residual_variance",),
        ),
        Query(
            title=f"Sample staleness — {policy_id}",
            x=("iter",),
            y=base + ("sample_staleness",),
        ),
        Query(
            title=f"Policy loss — {policy_id}",
            x=("iter",),
            y=base + ("policy_loss",),
        ),
        Query(
            title=f"Policy entropy — {policy_id}",
            x=("iter",),
            y=base + ("policy_entropy",),
        ),
        Query(
            title=f"Policy entropy coefficient — {policy_id}",
            x=("iter",),
            y=base + ("policy_entropy_coeff",),
        ),
        Query(
            title=f"Policy relative entropy — {policy_id}",
            x=("iter",),
            y=base + ("policy_relative_entropy",),
        ),
        Query(
            title=f"Entropy pressure — {policy_id}",
            x=("iter",),
            y=base + ("entropy_pressure",),
        ),
        Query(
            title=f"Policy KL — {policy_id}",
            x=("iter",),
            y=base + ("policy_kl",),
        ),
        Query(
            title=f"Policy KL coefficient — {policy_id}",
            x=("iter",),
            y=base + ("policy_kl_coeff",),
        ),
        Query(
            title=f"Value loss — {policy_id}",
            x=("iter",),
            y=base + ("value_loss",),
        ),
        Query(
            title=f"Value mean — {policy_id}",
            x=("iter",),
            y=base + ("value_mean",),
        ),
        Query(
            title=f"Value target — {policy_id}",
            x=("iter",),
            y=base + ("value_target",),
        ),
        Query(
            title=f"Gradient norm — {policy_id}",
            x=("iter",),
            y=base + ("gradient_norm",),
        ),
        Query(
            title=f"Gradient noise — {policy_id}",
            x=("iter",),
            y=base + ("gradient_noise",),
        ),
    ]


# Example only: substitute an ID that exists in the run.
EXAMPLE_POLICY_ID = "fisher_policy_m0_s3444837047"
EXAMPLE_POLICY_QUERIES = ray_policy_queries(EXAMPLE_POLICY_ID)


 Target wildcard API:



 ```python

 Query(

     title="Policy loss by policy",

     x=("iter",),

     y=("train", "learner", "by_policy", "*", "policy_loss"),

 )

 ```



 Expected result: one trace per policy ID.



 The same wildcard behavior is required for entropy, KL, value loss,

 gradient norm, and the other learner leaves.

 ## 10. Current concrete mechanism/seed/episode queries



 `RaySchema` stores:



 ```

 rollout

 └── by_mechanism[mechanism_id]

     └── by_seed[seed_id]

         └── by_episode[episode_id]

             └── FisheryMetricSchema

 ```



 Therefore a query for one known runtime episode can be generated as follows:

In [ ]:
def ray_episode_queries(
    *,
    phase: str,
    mechanism_id: str,
    seed_id: str,
    episode_id: str,
) -> list[Query]:
    base = (
        phase,
        "rollout",
        "by_mechanism",
        mechanism_id,
        "by_seed",
        seed_id,
        "by_episode",
        episode_id,
    )

    return [
        Query(
            title=f"{phase}: fish biomass m{mechanism_id} seed={seed_id}",
            x=("iter",),
            y=base + ("fish_norm",),
        ),
        Query(
            title=f"{phase}: realized harvest m{mechanism_id} seed={seed_id}",
            x=("iter",),
            y=base + ("H_realized",),
        ),
        Query(
            title=f"{phase}: reward m{mechanism_id} seed={seed_id}",
            x=("iter",),
            y=base + ("reward_mean",),
        ),
    ]


 ## 11. IMPORTANT schema mismatch in the proposed seed-aggregate query



 The requested example was:



 ```python

 Query(

     title="Fish biomass — mechanism 0",

     x=("iter",),

     y=(

         ("by_mechanism", "0", "by_seed", "100", "aggregate", "fish_norm"),

         ("by_mechanism", "0", "by_seed", "200", "aggregate", "fish_norm"),

         ("by_mechanism", "0", "by_seed", "300", "aggregate", "fish_norm"),

     ),

     reduce="mean",

     error="std",

 )

 ```



 But the supplied `SeedRolloutSchema` currently contains:



 ```python

 class SeedRolloutSchema(MetricSchema):

     by_episode: dict[EpisodeID, EpisodeRolloutSchema]

 ```



 It does **not** currently declare `aggregate`.



 There are two valid implementation choices:



 1. add:



    ```python

    aggregate: EpisodeRolloutSchema

    ```



    to `SeedRolloutSchema`, and populate it in the Ray adaptor; or



 2. keep the existing schema and make wildcard Query resolution aggregate the

    matching `by_episode` leaves.



 Do not document `by_seed/.../aggregate/...` as a working path until one of

 these two choices is implemented.

 ## 12. Target wildcard queries: mechanism ±std across seeds



 This section defines the **desired public Query syntax**. These queries should

 not be enabled in production until wildcard resolution is implemented.



 Minimum wildcard behavior:



 - `"*"` may match runtime keys in dynamic dict nodes;

 - matching order must be deterministic;

 - wildcard bindings must be preserved so x/y series remain aligned;

 - for mechanism × seed queries, the reporter must preserve mechanism as a

   plotted group while reducing across seeds;

 - `reduce="mean", error="std"` must produce one mean line plus ±1 std per

   mechanism.



 Desired query:

In [ ]:
TARGET_MECHANISM_STD_QUERY = Query(
    title="Train fish biomass by mechanism ±1 std across seeds",
    x=("iter",),
    y=(
        "train",
        "rollout",
        "by_mechanism",
        "*",
        "by_seed",
        "*",
        "by_episode",
        "*",
        "fish_norm",
    ),
    reduce="mean",
    error="std",
)


 Expected rendering:



 ```

 mechanism 0 mean + shaded ±1 std

 mechanism 1 mean + shaded ±1 std

 mechanism 2 mean + shaded ±1 std

 ...

 ```



 The reduction dimension is seed/episode; mechanism remains a grouping

 dimension.



 If the implementation instead adds `SeedRolloutSchema.aggregate`, the

 shorter target path becomes:



 ```python

 (

     "train",

     "rollout",

     "by_mechanism",

     "*",

     "by_seed",

     "*",

     "aggregate",

     "fish_norm",

 )

 ```

 ## 13. Target wildcard query: train-vs-eval shaded mechanism plots



 The dev visualization plotted:



 - train and eval in the same figure;

 - one curve per mechanism;

 - mean across seeds;

 - ±1 standard deviation shading;

 - x-axis = horizon step for horizon views, or training iteration for

   over-training views.



 Desired query-level expression:

In [ ]:
TARGET_TRAIN_EVAL_FISH_QUERY = Query(
    title="Fish biomass: train vs eval by mechanism ±1 std across seeds",
    x=("iter",),
    y=(
        (
            "train",
            "rollout",
            "by_mechanism",
            "*",
            "by_seed",
            "*",
            "by_episode",
            "*",
            "fish_norm",
        ),
        (
            "eval",
            "rollout",
            "by_mechanism",
            "*",
            "by_seed",
            "*",
            "by_episode",
            "*",
            "fish_norm",
        ),
    ),
    reduce="mean",
    error="std",
)


 The renderer must preserve both:



 ```

 phase     = train | eval

 mechanism = runtime mechanism ID

 ```



 and reduce only across the seed/episode replicates.



 Required visual parity with dev:



 - train and eval visually distinguishable;

 - mechanism identity visually distinguishable;

 - mean line;

 - ±1 std shaded region;

 - consistent trace labels;

 - deterministic ordering.

 ## 14. ES queries: current fishery acceptance test



 The current fishery optimization example uses four candidate positions and

 optimizes:



 ```

 fixed_quota

 restoration_subsidy

 ```



 The following constants make the expected regression suite explicit.

In [ ]:
ES_CANDIDATE_IDS = ("0", "1", "2", "3")
ES_PARAMETER_NAMES = ("fixed_quota", "restoration_subsidy")


 ### 14.1 Fitness over generations



 This is the data needed to reproduce the dev figure:



 - all candidate fitness values;

 - generation mean;

 - generation best;

 - global best.

In [ ]:
ES_FITNESS_QUERIES = [
    Query(
        title="Fitness over outer optimization iterations",
        x=("generation",),
        y=(
            ("by_mechanism", "0", "fitness"),
            ("by_mechanism", "1", "fitness"),
            ("by_mechanism", "2", "fitness"),
            ("by_mechanism", "3", "fitness"),
            ("fitness_mean",),
            ("fitness_best",),
        ),
    ),
    Query(
        title="Global best fitness",
        x=("generation",),
        y=("best_fitness_global",),
    ),
    Query(
        title="ES sigma",
        x=("generation",),
        y=("sigma",),
    ),
    Query(
        title="ES population size",
        x=("generation",),
        y=("population_size",),
    ),
]


 The data is available with ordinary line queries, but exact dev visual parity

 still requires trace styling:



 - candidate values = markers;

 - generation mean = line + markers;

 - generation best = line + markers.



 That trace-mode distinction is a renderer/query-style TODO.

 ### 14.2 Search mean

In [ ]:
ES_SEARCH_MEAN_QUERY = Query(
    title="ES search mean",
    x=("generation",),
    y=(
        ("search_mean", "fixed_quota", "value"),
        ("search_mean", "restoration_subsidy", "value"),
    ),
)


 ### 14.3 Global-best parameter values

In [ ]:
ES_GLOBAL_BEST_QUERY = Query(
    title="Global-best mechanism parameters",
    x=("generation",),
    y=(
        ("global_best", "fixed_quota", "value"),
        ("global_best", "restoration_subsidy", "value"),
    ),
)


 ### 14.4 Fitness versus parameter: current concrete workaround



 Today, one Query has one x path. Therefore the simplest working smoke test is

 one query per candidate position and parameter.

In [ ]:
def es_parameter_fitness_queries(
    *,
    candidate_ids: tuple[str, ...] = ES_CANDIDATE_IDS,
    parameter_names: tuple[str, ...] = ES_PARAMETER_NAMES,
) -> list[Query]:
    queries: list[Query] = []

    for parameter_name in parameter_names:
        for candidate_id in candidate_ids:
            queries.append(
                Query(
                    title=f"Fitness vs {parameter_name} — candidate {candidate_id}",
                    x=(
                        "by_mechanism",
                        candidate_id,
                        "by_parameter",
                        parameter_name,
                        "value",
                    ),
                    y=(
                        "by_mechanism",
                        candidate_id,
                        "fitness",
                    ),
                )
            )

    return queries


ES_PARAMETER_FITNESS_QUERIES = es_parameter_fitness_queries()


 Exact dev parity requires all candidate positions in **one cumulative scatter

 plot per parameter**, not one figure per candidate.



 Desired target wildcard concept:



 ```python

 Query(

     title="Fitness vs fixed_quota",

     x=(

         "by_mechanism",

         "*",

         "by_parameter",

         "fixed_quota",

         "value",

     ),

     y=("by_mechanism", "*", "fitness"),

 )

 ```



 The wildcard binding for `by_mechanism/*` must be paired between x and y so

 candidate `i`'s parameter value always pairs with candidate `i`'s fitness.



 Exact old dev styling also colors points by generation and includes

 generation/mechanism in hover metadata. That requires a scatter-capable

 query/renderer extension; plain x/y series selection is not sufficient.

 ### 14.5 Parallel coordinates



 The old dev ES visualization also had a cumulative parallel-coordinates plot:



 ```

 fixed_quota | restoration_subsidy | ... | fitness

 ```



 Every line = one evaluated candidate from one outer generation.



 Fitness is:



 - the final parallel-coordinate axis; and

 - the line color.



 The current `Query(x, y)` abstraction cannot express a multidimensional

 table/parallel-coordinates plot. This requires either:



 1. a dedicated `ParallelCoordinatesQuery`; or

 2. a generic table/multidimensional query consumed by W&B/Plotly.



 Do not force this into an ordinary one-x/one-y line query.

 ### 14.6 Generation-best parameter series



 The old dev code logged each parameter of the best candidate **of the current

 generation** separately.



 The supplied `ESSchema` has:



 - `search_mean`

 - `global_best`



 but not `generation_best`.



 For exact parity add:



 ```python

 generation_best: dict[str, ESParameterSchema] = Field(

     default_factory=dict

 )

 ```



 and populate it from:



 ```python

 population[best_idx]

 ```



 Then query it over `generation`.

 ## 15. Complete current query bundles



 These are the query sets that can be attached immediately, excluding paths

 that need runtime IDs.

In [ ]:
RAY_QUERIES = [
    *RAY_ROLLOUT_QUERIES,
    *RAY_PERFORMANCE_QUERIES,
]

ES_QUERIES = [
    *ES_FITNESS_QUERIES,
    ES_SEARCH_MEAN_QUERY,
    ES_GLOBAL_BEST_QUERY,
    *ES_PARAMETER_FITNESS_QUERIES,
]


 Example configuration:



 ```python

 bilevel_cfg = (

     BilevelConfig()

     .reporter(

         config=WandbConfig(

             project="bilevel",

             ...

         )

     )

     .outer(

         ESConfig()

         ...

         .reporting(

             schema=ESSchema,

             queries=ES_QUERIES,

         )

     )

     .inner(

         APPOptimizerConfig()

         .environment(

             env=FisheryRegulatedEnv,

             ...,

             schema=FisheryMetricSchema,

             queries=FISHERY_ENV_QUERIES,

         )

         .reporting(

             schema=RaySchema,

             queries=RAY_QUERIES,

         )

     )

 )

 ```



 Add concrete `ray_policy_queries(policy_id)` while the Query API still

 requires explicit runtime policy IDs.



 Once wildcard support is implemented, the configuration should not need to

 know mechanism IDs, seed IDs, episode IDs, policy IDs, or agent IDs.

 ## 16. Expected reporting lifecycle



 ### Environment



 At episode completion:



 ```python

 metrics = env.logger.peek()

 env.reporter.report(metrics)   # horizon plot before destructive reduction

 reduced = env.logger.reduce()

 ```



 The reduced episode object can then be passed upward into RLlib.



 ### Ray/RLlib



 The adaptor turns raw RLlib results into `RaySchema`, pushes them into its

 metric logger, and reports the accumulated/appropriate view.



 ### ES



 One completed generation becomes one `ESSchema` payload:



 ```python

 self.logger.push_data(metrics)

 accumulated_metrics = self.logger.peek()

 self.reporting.report(accumulated_metrics)

 ```



 This is why W&B does not need its own `_ES_HISTORY_TABLES` as the source of

 truth. The metric logger owns history.

 ## 17. Regression validation checklist



 Run a deterministic small integration configuration:



 - 4 mechanisms/candidates;

 - 2 or 3 seeds;

 - short horizon;

 - short inner training;

 - at least 3 ES generations.



 Compare feature branch against dev.



 ### Environment



 - Fish biomass over horizon.

 - Fish stock / next fish stock.

 - Growth / growth noise.

 - Attempted / realized / allowed harvest.

 - Quota stress.

 - Total normalized usage.

 - Per-agent reward.

 - Per-agent action.

 - Per-agent requested/delivered harvest.

 - Per-agent quota violation / quota penalty / risk penalty.

 - Any named observation/info plots that existed on dev must have explicit

   schema fields if exact naming is required.



 ### Inner optimizer



 - Raw RLlib rollout reward mean/min/max.

 - Episode length.

 - Episode counts.

 - Train/eval rollout metrics.

 - Performance/throughput/timing.

 - Per-policy learner losses.

 - Entropy / KL / value / gradients.

 - Per-mechanism rollout values.

 - Mean ±1 std across seeds for each mechanism.

 - Train-vs-eval shaded mechanism figures.



 ### ES



 - Candidate fitness markers over generations.

 - Generation mean.

 - Generation best.

 - Global best.

 - Sigma.

 - Search mean per optimized parameter.

 - Global-best parameter value per optimized parameter.

 - Generation-best parameter value per optimized parameter.

 - Cumulative fitness-vs-parameter scatter per optimized parameter.

 - Cumulative parallel coordinates with fitness as final axis and color.



 The companion `VISUALIZATION_FEATURE_TODO.md` contains the implementation and

 unit-test acceptance criteria.

 ## 18. Backend status



 W&B is the backend that must first reach exact dev parity.



 The feature branch is **not complete** until:



 - CSV export is implemented and tested;

 - TensorBoard reporting is implemented and tested.



 Reporter backends should consume the same `MetricSchema + Query` contract.

 Backend-specific code should not change the schema or query semantics.

 ## 19. Design rule for future metrics



 Before adding a visualization, answer these questions:



 1. **What entity owns the metric?**

    - environment/episode;

    - agent;

    - policy/learner;

    - optimizer performance;

    - ES generation/candidate/parameter.



 2. **What should the reducer be?**

    - `SERIES` if history itself is the metric;

    - `MEAN`, `MIN`, `MAX`, `SUM`, `LAST`, etc. for compiled episode or

      iteration summaries.



 3. **Does the key exist at runtime?**

    - static field -> add it to the appropriate schema;

    - dynamic entity ID -> use a `dict[ID, MetricSchema]` branch and eventually

      query it with `"*"`.



 4. **Is the visualization one-dimensional or multidimensional?**

    - ordinary line/scatter -> `Query`;

    - grouped wildcard series -> Query wildcard/grouping extension;

    - parallel coordinates / table-style visualization -> dedicated

      multidimensional query support.



 This keeps collection, aggregation, selection, and presentation separate.